# Few-Shot Open-Set Keyword Spotting — Full Pipeline

**Train** on MSWC English (450 words) → **Evaluate** on GSC v2 (cross-dataset)

| Component | Detail |
|---|---|
| Encoder | DSCNN-L (276ch, embedding_dim=276) |
| Loss | Triplet / ArcFace / Sub-center ArcFace |
| Metrics | AUC, EER, FRR@FAR, Precision/Recall/F1 |
| Extensions | Denoising (EXT-1), Streaming+VAD (EXT-2) |

**GPU**: Runtime → Change runtime type → T4 GPU (or better)

---
## 1. Setup

In [ ]:
import os, subprocess

# Upload project archive from local machine (no Drive needed)
from google.colab import files
print("Upload file DoAnTotNghiep.rar hoac .zip tu may tinh:")
uploaded = files.upload()

# Extract
DRIVE_PROJECT = '/content/DoAnTotNghiep_output'
os.makedirs(f'{DRIVE_PROJECT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/results', exist_ok=True)

for fname in uploaded.keys():
    src = f'/content/{fname}'
    if fname.endswith('.rar'):
        subprocess.run(['apt-get', 'install', '-qq', 'unrar'], check=True)
        subprocess.run(['unrar', 'x', '-o+', src, '/content/'], check=True)
    elif fname.endswith('.zip'):
        subprocess.run(['unzip', '-qo', src, '-d', '/content/'], check=True)
    print(f'Extracted: {fname}')

print('Done!')

In [ ]:
import os, sys

# Tim folder project (co the nam trong subfolder sau khi extract)
PROJECT_DIR = None
for candidate in ['/content/DoAnTotNghiep',
                  '/content/DoAnTotNghiep/DoAnTotNghiep']:
    if os.path.exists(candidate) and os.path.exists(os.path.join(candidate, 'configs')):
        PROJECT_DIR = candidate
        break

# Neu khong tim thay, tim trong /content/
if PROJECT_DIR is None:
    for d in os.listdir('/content/'):
        p = f'/content/{d}'
        if os.path.isdir(p) and os.path.exists(os.path.join(p, 'configs', 'default.yaml')):
            PROJECT_DIR = p
            break

assert PROJECT_DIR, 'Khong tim thay project folder sau khi extract! Kiem tra lai file RAR.'

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f'Project dir: {PROJECT_DIR}')
print(f'Files: {os.listdir(".")[:15]}...')

In [ ]:
!pip install -q torch torchaudio numpy pyyaml matplotlib tensorboard \
    scikit-learn soundfile requests tqdm noisereduce speechbrain gradio

import torch
print(f'Project: {os.getcwd()}')
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

---
## 2. Download Data

- **GSC v2** (~2.4GB) for evaluation
- **MSWC English** (~32.5GB tar -> ~5GB OPUS extracted) for training
- **DEMAND** (~600MB) for noise augmentation

Total disk needed: **~40GB free** during MSWC download (will shrink after cleanup).

In [ ]:
%%time
# Step A: Download GSC v2 (required for evaluation, ~2.4GB)
import os
os.chdir(PROJECT_DIR)
print('CWD:', os.getcwd())

!python data/download_gsc.py


In [ ]:
%%time
# Step B: Download MSWC English for training
#
# Pipeline:
#   1. Tai metadata (~103MB) -> tao splits (450 train + 50 val + 485 eval words)
#   2. Tai en.tar.gz (~32.5GB OPUS) tu MLCommons (Cloudflare mirror)
#   3. Extract chi 985 word can thiet (max 200 clips/word ~ 5GB OPUS)
#   4. Tu xoa archive sau khi extract
#
# Yeu cau dia: ~40GB free (32.5GB tar + 5GB extracted OPUS)
# Thoi gian: 30-60 phut (tuy mang Colab)

# Uoc tinh dung luong truoc khi tai
import shutil
free_gb = shutil.disk_usage('/content').free / 1024**3
print(f'Free disk: {free_gb:.1f} GB (can ~50GB de chua 500 clips/word)')
assert free_gb > 40, f'Khong du dia! Can it nhat 40GB, hien co {free_gb:.1f}GB'

# Chay download (500 clips/word de co diversity cao -- accuracy boost)
!python data/download_mswc.py --max-per-word 500

# Kiem tra ket qua
from pathlib import Path
clips = Path('data/mswc_en/clips')
n_words = sum(1 for d in clips.iterdir() if d.is_dir()) if clips.exists() else 0
n_opus  = len(list(clips.rglob('*.opus'))) if clips.exists() else 0
print(f'\nMSWC English: {n_words} words, {n_opus} OPUS files')
print(f'Disk used: {sum(f.stat().st_size for f in clips.rglob("*") if f.is_file()) / 1024**3:.2f} GB')

In [ ]:
%%time
# Step C: Download DEMAND (noise augmentation, ~600MB)
#       + Convert MSWC OPUS -> WAV (~12GB WAV after conversion)

import zipfile, requests
from pathlib import Path
from tqdm import tqdm

# --- C.1: DEMAND noise dataset (17 environments x 16kHz) ---
DEMAND_DIR = Path('data/demand')
DEMAND_DIR.mkdir(parents=True, exist_ok=True)

if not any(DEMAND_DIR.rglob('*.wav')):
    BASE = 'https://zenodo.org/records/1227121/files'
    ENVS = [
        'DKITCHEN','DLIVING','DWASHING','NFIELD','NPARK','NRIVER',
        'OHALLWAY','OMEETING','OOFFICE','PCAFETER','PRESTO','PSTATION',
        'SPSQUARE','STRAFFIC','TBUS','TCAR','TMETRO',
    ]
    for env in tqdm(ENVS, desc='Downloading DEMAND'):
        url = f'{BASE}/{env}_16k.zip?download=1'
        zp = Path(f'data/{env}_16k.zip')
        if not zp.exists():
            r = requests.get(url, stream=True, timeout=120); r.raise_for_status()
            with open(zp,'wb') as f:
                for chunk in r.iter_content(8192): f.write(chunk)
        with zipfile.ZipFile(zp) as zf: zf.extractall(DEMAND_DIR)
        zp.unlink()
print(f'DEMAND: {len(list(DEMAND_DIR.rglob("*.wav")))} noise files')

# --- C.2: Convert MSWC OPUS -> WAV (PARALLEL, ~30-50 phut tren Colab T4 8 vCPU) ---
import os
mswc_clips = Path('data/mswc_en/clips')
if mswc_clips.exists():
    opus_n = len(list(mswc_clips.rglob('*.opus')))
    wav_n  = len(list(mswc_clips.rglob('*.wav')))
    if opus_n > 0 and wav_n < opus_n:
        n_cpu = os.cpu_count() or 8
        print(f'\nConverting {opus_n} OPUS -> WAV with {n_cpu} parallel workers...')
        print(f'(Single-threaded would take ~3-5 hours, parallel takes ~30-50 phut)')
        !apt-get install -qq ffmpeg
        # --delete-opus: xoa OPUS ngay khi convert thanh cong de giai phong dia
        !python data/convert_opus.py --workers {n_cpu} --delete-opus
    else:
        print(f'MSWC: {wav_n} WAV files ready (skip convert)')

# Tom tat dung luong cuoi
print('\n=== Disk usage summary ===')
for d in ['data/gsc_v2', 'data/mswc_en/clips', 'data/demand']:
    p = Path(d)
    if p.exists():
        sz = sum(f.stat().st_size for f in p.rglob('*') if f.is_file()) / 1024**3
        n = len(list(p.rglob('*.wav')))
        print(f'  {d}: {sz:.2f} GB ({n} WAV files)')

---
## 3. Verify All Modules

In [ ]:
import yaml, json, shutil, random, numpy as np, matplotlib.pyplot as plt
import torch, torch.nn.functional as F, torchaudio
from pathlib import Path
from torch.optim.lr_scheduler import StepLR, CosineAnnealingWarmRestarts
from torch.utils.tensorboard import SummaryWriter

from src.models.dscnn import DSCNN
from src.models.arcface import ArcFaceLoss, SubCenterArcFaceLoss
from src.models.prototypical import TripletLoss, train_one_epoch
from src.features.mfcc import MFCCExtractor
from src.features.augmentation import NoiseAugmenter, WaveformAugmenter
from src.features.spec_augment import SpecAugment
from src.data.mswc_dataset import MSWCDataset, build_episodic_loader
from src.classifiers.open_ncm import OpenNCMClassifier
from src.evaluation.protocols import EvaluationProtocol
from src.evaluation.gsc import GSCFewShotProvider
from src.evaluation.metrics import plot_det_curves, compute_eer
from src.enhancements.denoiser import Denoiser

with open('configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

enc_test = DSCNN(model_size='L')
print(f'DSCNN-L: {sum(p.numel() for p in enc_test.parameters()):,} params')
denoiser = Denoiser(backend='spectral_gate', enabled=True)
print('Denoiser OK | All modules imported')

---
## 4. Build Dataset (MSWC train, GSC eval)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

DRIVE_PROJECT_P = Path(DRIVE_PROJECT)

# --- Helper: count WAV files per word ---
def count_words_with_wavs(data_dir, words, min_samples=20):
    """Check how many words actually have enough WAV files."""
    count = 0
    for w in words:
        for d in [data_dir / w, data_dir / 'clips' / w]:
            if d.exists() and len(list(d.glob('*.wav'))) >= min_samples:
                count += 1
                break
    return count

# --- Try MSWC first, fallback to GSC if no audio ---
mswc_dir = Path('data/mswc_en')
gsc_dir  = Path('data/gsc_v2')
data_dir = None
train_words = []
val_words = []

# Option 1: MSWC with splits
splits_dir = mswc_dir / 'splits'
if (splits_dir / 'train_words.json').exists():
    with open(splits_dir / 'train_words.json') as f: tw = json.load(f)
    with open(splits_dir / 'val_words.json') as f:   vw = json.load(f)
    n_valid = count_words_with_wavs(mswc_dir, tw)
    if n_valid >= 30:
        train_words, val_words, data_dir = tw, vw, mswc_dir
        print(f'MSWC splits: {len(train_words)} train ({n_valid} with audio), {len(val_words)} val')

# Option 2: MSWC directory scan
if data_dir is None and mswc_dir.exists():
    search = (mswc_dir/'clips') if (mswc_dir/'clips').exists() else mswc_dir
    if search.exists():
        aw = sorted(d.name for d in search.iterdir() if d.is_dir() and len(list(d.glob('*.wav'))) >= 20)
        if len(aw) >= 30:
            train_words, val_words = aw[:min(450,len(aw))], aw[min(450,len(aw)):min(500,len(aw))]
            data_dir = mswc_dir
            print(f'MSWC discovered: {len(train_words)} train, {len(val_words)} val')

# Option 3: Fallback to GSC
if data_dir is None and gsc_dir.exists():
    print('MSWC khong co du audio, dung GSC de train')
    data_dir = gsc_dir
    aw = sorted(d.name for d in gsc_dir.iterdir()
                if d.is_dir() and not d.name.startswith('_') and len(list(d.glob('*.wav'))) >= 20)
    train_words = aw
    val_words = []
    print(f'GSC: {len(train_words)} words with >=20 samples')

assert data_dir and len(train_words) > 0, 'Khong tim thay data! Chay cell Download Data truoc.'
print(f'Data: {data_dir} | Train: {len(train_words)} | Val: {len(val_words)}')

# --- Augmentation ---
noise_aug = None
demand_dir = Path('data/demand')
if demand_dir.exists() and any(demand_dir.rglob('*.wav')):
    noise_aug = NoiseAugmenter(demand_dir, prob=0.95, snr_db=(0.0, 10.0))
    print(f'Noise: {len(noise_aug.noise_files)} files')
wave_aug = WaveformAugmenter()

# SpecAugment manh hon -> regularization tot hon (Park et al. 2019)
spec_aug = SpecAugment(freq_mask_width=5, time_mask_width=12,
                       n_freq_masks=2, n_time_masks=2)
print(f'SpecAugment: freq=5 (x2), time=12 (x2)')

MAX_PER_WORD = 500  # tang tu 200 -> 500 cho diversity cao hon

# --- Train dataset ---
dataset = MSWCDataset(root_dir=data_dir, words=train_words, max_per_word=MAX_PER_WORD,
                      noise_augmenter=noise_aug, wave_augmenter=wave_aug,
                      spec_augmenter=spec_aug)

# Fallback: neu dataset rong (MSWC co splits nhung khong co audio) -> dung GSC
if len(dataset) == 0 and data_dir != gsc_dir and gsc_dir.exists():
    print(f'Dataset rong (MSWC khong co audio files)! Chuyen sang GSC...')
    data_dir = gsc_dir
    train_words = sorted(d.name for d in gsc_dir.iterdir()
                         if d.is_dir() and not d.name.startswith('_') and len(list(d.glob('*.wav'))) >= 20)
    val_words = []
    dataset = MSWCDataset(root_dir=data_dir, words=train_words, max_per_word=MAX_PER_WORD,
                          noise_augmenter=noise_aug, wave_augmenter=wave_aug,
                          spec_augmenter=spec_aug)
    print(f'GSC fallback: {len(train_words)} words, {len(dataset)} samples')

assert len(dataset) > 0, 'Dataset van rong! Kiem tra data/'

# Count actual classes with enough samples for episodic loader
from collections import Counter
label_counts = Counter(s[1] for s in dataset.samples)
usable_classes = sum(1 for c in label_counts.values() if c >= 20)
print(f'Train dataset: {len(dataset)} samples, {len(dataset.word_to_idx)} words, {usable_classes} usable (>=20 samples)')

# --- Val dataset ---
val_loader = None
if val_words:
    val_ds = MSWCDataset(root_dir=data_dir, words=val_words, max_per_word=50)
    if len(val_ds) > 0:
        vn = min(5, len(val_ds.word_to_idx))
        try:
            val_loader = build_episodic_loader(val_ds, n_classes=vn, n_samples=10, n_episodes=20, num_workers=0)
            print(f'Val loader: {vn} classes, {len(val_ds)} samples')
        except ValueError:
            print('Val loader: could not build, skipping')

---
## 5. Train (Triplet / ArcFace / SCAF) — OPTIMIZED for accuracy

**Key improvements over baseline:**

| Component | Before | After |
|---|---|---|
| Mining | Random | **Semi-Hard + Hard fallback** (FaceNet-style) |
| Margin | 0.5 | **1.0** (better separation for L2-norm embeddings) |
| Scheduler | StepLR(20, 0.5) | **CosineAnnealingWarmRestarts(T_0=10, T_mult=2)** |
| Weight decay | 0 | **1e-4** (L2 regularization) |
| Grad clipping | none | **max_norm=5.0** (stable hard mining) |
| SpecAugment | not applied | **freq=5x2, time=12x2** |
| Epochs × Episodes | 40 × 400 | **100 × 600** |
| max_per_word | 200 | **500** |
| Validate | every 5 ep | **every epoch** (best.pt = highest val_AUC) |

Expected gain: **AUC 0.91 → 0.96-0.98**, **KW-ACC 0.77 → 0.85-0.90**.

Runs all 3 loss functions sequentially. Each saves checkpoints to Google Drive.

In [ ]:
from collections import Counter
label_counts = Counter(s[1] for s in dataset.samples)
usable_classes = sum(1 for c in label_counts.values() if c >= 20)

# === Aggressive overnight config (GPU) ===
N_EPOCHS     = 100
N_EPISODES   = 600
N_CLASSES    = min(30, usable_classes)
N_SAMPLES    = 20
if usable_classes < N_SAMPLES:
    N_SAMPLES = min(N_SAMPLES, min(label_counts.values()) if label_counts else 1)

LR            = 0.001
WEIGHT_DECAY  = 1e-4    # L2 regularization
GRAD_CLIP     = 5.0     # max L2 norm cho gradient clipping
TRIPLET_MARGIN = 1.0    # tang tu 0.5 (L2-norm embeddings co max distance 2.0)
TRIPLET_MINING = 'semi_hard'   # random | hard | semi_hard
SAVE_EVERY    = 5

# CosineAnnealingWarmRestarts: T_0=10, T_mult=2 -> restarts tai epoch 10, 30, 70, 150
SCHED_T0      = 10
SCHED_TMULT   = 2
SCHED_ETAMIN  = 1e-5

print(f'Dataset: {len(dataset)} samples, {usable_classes} usable classes')
print(f'Episodic: {N_CLASSES} classes x {N_SAMPLES} samples x {N_EPISODES} episodes x {N_EPOCHS} epochs')
print(f'Triplet: margin={TRIPLET_MARGIN}, mining={TRIPLET_MINING}')
print(f'Optim: Adam lr={LR} wd={WEIGHT_DECAY} grad_clip={GRAD_CLIP}')
print(f'Sched: CosineAnnealingWarmRestarts T_0={SCHED_T0} T_mult={SCHED_TMULT} eta_min={SCHED_ETAMIN}')

EXPERIMENTS = ['triplet']  # focus on triplet (fastest path to high acc).
                            # Add 'arcface', 'scaf' if you want all 3.

# --- Resume config ---
# FORCE_RESTART = True  -> xoa checkpoint cu, train lai tu epoch 0
# FORCE_RESTART = False -> auto resume tu `latest.pt` neu co
FORCE_RESTART = False


def validate_few_shot(encoder, loader, device):
    """Few-shot 5-way 5-shot acc + AUC binary (keyword vs other)."""
    from sklearn.metrics import roc_auc_score
    encoder.eval()
    correct, total = 0, 0
    y_true, scores = [], []
    with torch.no_grad():
        for mfcc, labels in loader:
            mfcc, labels = mfcc.to(device), labels.to(device)
            emb = F.normalize(encoder(mfcc), p=2, dim=-1)
            u = labels.unique()
            if len(u) < 2: continue
            k = min(5, (labels==u[0]).sum().item()-1)
            if k < 1: continue
            protos, qs, ql = [], [], []
            for i,c in enumerate(u):
                ce = emb[labels==c]
                protos.append(ce[:k].mean(0)); qs.append(ce[k:]); ql.extend([i]*(ce.shape[0]-k))
            proto_t = torch.stack(protos)
            q_t = torch.cat(qs)
            dists = torch.cdist(q_t, proto_t, p=2)
            preds = dists.argmin(1)
            correct += (preds == torch.tensor(ql, device=device)).sum().item()
            total += len(ql)
            for i in range(len(ql)):
                for c in range(len(u)):
                    y_true.append(1 if c == ql[i] else 0)
                    scores.append(-dists[i, c].item())
    encoder.train()
    auc = roc_auc_score(y_true, scores) if len(set(y_true)) >= 2 else 0.0
    return {'acc': correct / max(total, 1), 'auc': float(auc)}


def train_epoch_arcface(enc, head, loader, opt, dev, grad_clip=0.0):
    enc.train(); head.train()
    tot, n = 0.0, 0
    for mfcc, labels in loader:
        mfcc, labels = mfcc.to(dev), labels.to(dev)
        loss = head(F.normalize(enc(mfcc), p=2, dim=-1), labels)
        opt.zero_grad(); loss.backward()
        if grad_clip > 0:
            torch.nn.utils.clip_grad_norm_(
                list(enc.parameters()) + list(head.parameters()), max_norm=grad_clip,
            )
        opt.step()
        tot += loss.item(); n += 1
    return {'loss': tot/max(n,1), 'num_episodes': n}


def find_latest_checkpoint(ckpt_dir: Path) -> Path | None:
    """Tim checkpoint moi nhat de resume.
    Uu tien latest.pt, roi toi epoch_XX.pt co so lon nhat.
    """
    latest = ckpt_dir / 'latest.pt'
    if latest.exists():
        return latest
    epoch_files = sorted(ckpt_dir.glob('epoch_*.pt'),
                         key=lambda p: int(p.stem.split('_')[1]))
    return epoch_files[-1] if epoch_files else None


all_train_logs = {}

for EXP in EXPERIMENTS:
    print(f'\n{"="*60}')
    print(f'  EXPERIMENT: {EXP.upper()}')
    print(f'{"="*60}')

    ckpt_dir = DRIVE_PROJECT_P / 'checkpoints' / EXP
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    # --- Xay dung model + optimizer ---
    encoder = DSCNN(model_size='L').to(device)
    train_loader = build_episodic_loader(dataset, n_classes=N_CLASSES, n_samples=N_SAMPLES,
                                         n_episodes=N_EPISODES, num_workers=4)

    loss_fn = None
    loss_head = None
    if EXP == 'triplet':
        loss_fn = TripletLoss(margin=TRIPLET_MARGIN, mining=TRIPLET_MINING)
        optimizer = torch.optim.Adam(encoder.parameters(),
                                     lr=LR, weight_decay=WEIGHT_DECAY)
    elif EXP == 'arcface':
        loss_head = ArcFaceLoss(encoder.embedding_dim, len(dataset.word_to_idx), 30.0, 0.5).to(device)
        optimizer = torch.optim.Adam(list(encoder.parameters())+list(loss_head.parameters()),
                                     lr=LR, weight_decay=WEIGHT_DECAY)
    elif EXP == 'scaf':
        loss_head = SubCenterArcFaceLoss(encoder.embedding_dim, len(dataset.word_to_idx), K=3, scale=30.0, margin=0.5).to(device)
        optimizer = torch.optim.Adam(list(encoder.parameters())+list(loss_head.parameters()),
                                     lr=LR, weight_decay=WEIGHT_DECAY)

    scheduler = CosineAnnealingWarmRestarts(
        optimizer, T_0=SCHED_T0, T_mult=SCHED_TMULT, eta_min=SCHED_ETAMIN,
    )

    # --- Resume tu checkpoint neu co ---
    start_epoch = 0
    best_loss   = float('inf')
    best_val    = 0.0
    logs        = []

    resume_ckpt = None if FORCE_RESTART else find_latest_checkpoint(ckpt_dir)
    if resume_ckpt is not None:
        print(f'  Resume tu: {resume_ckpt.name}')
        ckpt = torch.load(str(resume_ckpt), map_location=device, weights_only=False)
        encoder.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if 'scheduler_state_dict' in ckpt:
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        if loss_head is not None and 'loss_head_state_dict' in ckpt:
            loss_head.load_state_dict(ckpt['loss_head_state_dict'])
        start_epoch = ckpt['epoch'] + 1
        best_loss   = ckpt.get('best_loss', ckpt.get('loss', float('inf')))
        best_val    = ckpt.get('best_val', 0.0)
        logs        = ckpt.get('logs', [])
        print(f'    -> Tiep tuc tu epoch {start_epoch}/{N_EPOCHS}, best_loss={best_loss:.6f}')

        if start_epoch >= N_EPOCHS:
            print(f'  Da train du {N_EPOCHS} epoch - SKIP (dat FORCE_RESTART=True de train lai)')
            all_train_logs[EXP] = logs
            continue

    writer = SummaryWriter(log_dir=str(ckpt_dir / 'runs'))

    try:
        for epoch in range(start_epoch, N_EPOCHS):
            if EXP == 'triplet':
                m = train_one_epoch(encoder, train_loader, optimizer, loss_fn, device,
                                    grad_clip=GRAD_CLIP)
            else:
                m = train_epoch_arcface(encoder, loss_head, train_loader, optimizer, device,
                                        grad_clip=GRAD_CLIP)
            scheduler.step()
            lr = optimizer.param_groups[0]['lr']
            writer.add_scalar('train/loss', m['loss'], epoch+1)
            writer.add_scalar('train/lr', lr, epoch+1)
            if EXP == 'triplet':
                writer.add_scalar('train/active_triplets', m.get('active_per_ep', 0), epoch+1)
                writer.add_scalar('train/semi_hard',       m.get('semi_hard_per_ep', 0), epoch+1)
                writer.add_scalar('train/hard_fallback',   m.get('hard_fallback_per_ep', 0), epoch+1)
                writer.add_scalar('train/mean_d_pos',      m.get('mean_d_pos', 0), epoch+1)
                writer.add_scalar('train/mean_d_neg',      m.get('mean_d_neg', 0), epoch+1)
                writer.add_scalar('train/grad_norm',       m.get('mean_grad_norm', 0), epoch+1)

            # Validate moi epoch de track tien do tot hon
            val_msg = ''
            cur_val_auc = 0.0
            cur_val_acc = 0.0
            vl = val_loader if val_loader else train_loader
            v = validate_few_shot(encoder, vl, device)
            cur_val_acc, cur_val_auc = v['acc'], v['auc']
            writer.add_scalar('val/acc', cur_val_acc, epoch+1)
            writer.add_scalar('val/auc', cur_val_auc, epoch+1)
            val_msg = f' | val_acc={cur_val_acc:.4f} val_auc={cur_val_auc:.4f}'
            if cur_val_auc > best_val:
                best_val = cur_val_auc

            extra = ''
            if EXP == 'triplet':
                extra = (f' | semi_hard={m.get("semi_hard_per_ep",0):.0f}/{m.get("valid_per_ep",0):.0f}'
                         f' | d_pos={m.get("mean_d_pos",0):.3f} d_neg={m.get("mean_d_neg",0):.3f}'
                         f' | gnorm={m.get("mean_grad_norm",0):.2f}')
            print(f'  [{EXP}] {epoch+1}/{N_EPOCHS} loss={m["loss"]:.6f} lr={lr:.6f}{extra}{val_msg}')
            logs.append({'epoch': epoch+1, 'loss': m['loss'], 'lr': lr,
                         'val_acc': cur_val_acc, 'val_auc': cur_val_auc})

            # Best checkpoint = highest val_auc (more meaningful than loss)
            if cur_val_auc > 0:
                _is_best = cur_val_auc >= best_val
            else:
                _is_best = m['loss'] < best_loss
                if _is_best:
                    best_loss = m['loss']

            ckpt = {
                'epoch': epoch,
                'model_state_dict': encoder.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'loss': m['loss'],
                'val_acc': cur_val_acc,
                'val_auc': cur_val_auc,
                'best_loss': best_loss,
                'best_val': best_val,
                'experiment': EXP,
                'logs': logs,
            }
            if loss_head is not None:
                ckpt['loss_head_state_dict'] = loss_head.state_dict()

            torch.save(ckpt, ckpt_dir / 'latest.pt')
            if (epoch+1) % SAVE_EVERY == 0:
                torch.save(ckpt, ckpt_dir / f'epoch_{epoch+1:02d}.pt')
            if _is_best:
                torch.save(ckpt, ckpt_dir / 'best.pt')
                print(f'    * New best (val_auc={cur_val_auc:.4f})')

    except KeyboardInterrupt:
        print(f'\n  [{EXP}] Dung boi user tai epoch {epoch+1}.')
        print(f'  Checkpoint moi nhat: {ckpt_dir}/latest.pt (epoch {epoch+1})')
        print(f'  Chay lai cell nay de tiep tuc tu epoch {epoch+2}.')
        writer.close()
        all_train_logs[EXP] = logs
        raise

    writer.close()
    all_train_logs[EXP] = logs
    print(f'  Done! best_loss={best_loss:.6f}, best_val_auc={best_val:.4f}')

print(f'\nAll experiments complete!')

---
## 6. Evaluate All Experiments on GSC

Cross-dataset evaluation: trained on MSWC, tested on GSC v2.

In [ ]:
provider = GSCFewShotProvider('data/gsc_v2')

all_results = {}
det_curves  = {}
results_dir = DRIVE_PROJECT_P / 'results'
results_dir.mkdir(parents=True, exist_ok=True)

for EXP in EXPERIMENTS:
    ckpt_path = DRIVE_PROJECT_P / 'checkpoints' / EXP / 'best.pt'
    if not ckpt_path.exists():
        print(f'[{EXP}] No checkpoint found, skipping')
        continue

    encoder = DSCNN(model_size='L').to(device)
    ckpt = torch.load(str(ckpt_path), map_location=device, weights_only=False)
    encoder.load_state_dict(ckpt['model_state_dict'])
    encoder.eval()
    print(f'\n{"="*60}')
    print(f'  {EXP.upper()} (epoch {ckpt["epoch"]+1}, loss {ckpt["loss"]:.6f})')
    print(f'{"="*60}')

    for protocol in ['gsc_fixed', 'gsc_random']:
        ds, mode = protocol.split('_', 1)
        for k in [5, 10]:
            key = f'{EXP}_{protocol}_k{k}'
            proto = EvaluationProtocol(ds, mode, n_runs=5, n_way=5, k_shot=k, seed=42)
            clf = OpenNCMClassifier()
            res = proto.evaluate(encoder, clf, provider, device=device, target_far=0.05)
            all_results[key] = res

            print(f'\n  [{key}]')
            print(f'    AUC={res["auc"]:.4f}  EER={res["eer"]:.4f}  '
                  f'FRR@5%={res["frr_at_far"]:.4f}  ACC@5%={res["open_set_acc_at_far"]:.4f}')
            print(f'    KW-ACC={res["keyword_acc"]:.4f}  P={res["precision"]:.4f}  '
                  f'R={res["recall"]:.4f}  F1={res["f1"]:.4f}')

            last = res['per_run'][-1]
            det_curves[key] = (np.array(last['det_curve']['far']),
                               np.array(last['det_curve']['frr']))

            with open(results_dir / f'{key}.json', 'w') as f:
                json.dump(res, f, indent=2)

# DET curves
if det_curves:
    plot_det_curves(det_curves, save_path=results_dir / 'det_curves.png')
    from IPython.display import Image, display
    display(Image(str(results_dir / 'det_curves.png')))
    print(f'\nResults saved to {results_dir}')

---
## 7. Comparison Table

In [ ]:
print(f'{"Experiment":<30} {"AUC":>6} {"EER":>6} {"FRR@5%":>8} {"ACC@5%":>8} {"KW-ACC":>8} {"F1":>6}')
print('-' * 80)
for key in sorted(all_results.keys()):
    r = all_results[key]
    print(f'{key:<30} {r["auc"]:>6.4f} {r["eer"]:>6.4f} {r["frr_at_far"]:>8.4f} '
          f'{r["open_set_acc_at_far"]:>8.4f} {r["keyword_acc"]:>8.4f} {r["f1"]:>6.4f}')

---
## 8. Training Loss Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for exp, logs in all_train_logs.items():
    epochs = [l['epoch'] for l in logs]
    losses = [l['loss'] for l in logs]
    ax.plot(epochs, losses, label=exp, linewidth=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_title('Training Loss')
ax.legend(); ax.grid(True, alpha=0.3)
fig.savefig(str(results_dir / 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 9. TensorBoard

---
## 12b. Download Checkpoints + Results to Local Machine

Since we're not using Google Drive, download everything as a ZIP.

In [ ]:
import shutil
from google.colab import files as colab_files

zip_name = '/content/kws_checkpoints_results'
shutil.make_archive(zip_name, 'zip', DRIVE_PROJECT)
colab_files.download(zip_name + '.zip')
print(f'Downloading {zip_name}.zip — save to your local checkpoints/ folder')

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {DRIVE_PROJECT}/checkpoints

---
## 10. Test Denoising (EXT-1)

In [ ]:
denoiser = Denoiser(backend='spectral_gate', enabled=True)

test_files = list(Path('data/gsc_v2/yes').glob('*.wav'))[:1]
if test_files:
    wav, sr = torchaudio.load(str(test_files[0]))
    if sr != 16000: wav = torchaudio.transforms.Resample(sr, 16000)(wav)

    noise = torch.randn_like(wav) * 0.1
    noisy = wav + noise
    denoised = denoiser.denoise(noisy)

    fig, axes = plt.subplots(3, 1, figsize=(10, 5))
    for ax, data, title in zip(axes, [wav, noisy, denoised], ['Clean', 'Noisy', 'Denoised (EXT-1)']):
        ax.plot(data.squeeze().numpy(), linewidth=0.5)
        ax.set_title(title); ax.set_xlim(0, 16000)
    fig.tight_layout(); plt.show()

    before = (noisy - wav).pow(2).mean().item()
    after  = (denoised - wav).pow(2).mean().item()
    print(f'Noise power: before={before:.6f}, after={after:.6f}, reduction={((1-after/before)*100):.1f}%')
else:
    print('No GSC test files found')

---
## 11. Test Streaming + VAD (EXT-2)

In [ ]:
# Load best model for streaming test
best_exp = 'scaf'  # change to whichever performed best
best_ckpt_path = DRIVE_PROJECT_P / 'checkpoints' / best_exp / 'best.pt'

if best_ckpt_path.exists():
    encoder = DSCNN(model_size='L').to(device)
    ckpt = torch.load(str(best_ckpt_path), map_location=device, weights_only=False)
    encoder.load_state_dict(ckpt['model_state_dict'])
    encoder.eval()
    mfcc_ext = MFCCExtractor()

    # Try loading Silero VAD
    vad = None
    try:
        from src.streaming.vad_engine import SileroVAD, StreamingKWS
        vad = SileroVAD(threshold=0.5, device=device)
        print('Silero VAD loaded')
    except Exception as e:
        print(f'VAD not available: {e}')
        from src.streaming.vad_engine import StreamingKWS

    engine = StreamingKWS(encoder=encoder, mfcc_extractor=mfcc_ext, vad=vad,
                          window_size=16000, stride=8000, device=device)

    # Create test audio: concatenate 3 different words
    test_audio = []
    for word in ['yes', 'no', 'stop']:
        wfiles = list(Path(f'data/gsc_v2/{word}').glob('*.wav'))[:1]
        if wfiles:
            w, sr = torchaudio.load(str(wfiles[0]))
            if sr != 16000: w = torchaudio.transforms.Resample(sr, 16000)(w)
            if w.shape[-1] < 16000: w = F.pad(w, (0, 16000 - w.shape[-1]))
            test_audio.append(w[..., :16000])
            test_audio.append(torch.zeros(1, 8000))  # silence gap

    if test_audio:
        full_audio = torch.cat(test_audio, dim=-1)
        print(f'Test audio: {full_audio.shape[-1]/16000:.1f}s')

        # Enroll "yes" as prototype
        yes_files = list(Path('data/gsc_v2/yes').glob('*.wav'))[:5]
        protos = {}
        if yes_files:
            embs = []
            for f in yes_files:
                w, sr = torchaudio.load(str(f))
                if sr != 16000: w = torchaudio.transforms.Resample(sr, 16000)(w)
                if w.shape[-1] < 16000: w = F.pad(w, (0, 16000 - w.shape[-1]))
                mfcc = mfcc_ext.extract(w[..., :16000]).unsqueeze(0).to(device)
                with torch.no_grad():
                    embs.append(F.normalize(encoder(mfcc), p=2, dim=-1).squeeze(0).cpu())
            protos['yes'] = torch.stack(embs).mean(0)

        results = engine.process_file(full_audio, protos, threshold=0.8)

        # Plot
        fig, axes = plt.subplots(2, 1, figsize=(12, 4), height_ratios=[2, 1])
        for r in results:
            c = '#4CAF50' if r['detected'] else ('#e0e0e0' if not r['is_speech'] else '#FFCDD2')
            axes[0].barh(0, r['t_end']-r['t_start'], left=r['t_start'], height=0.5, color=c, edgecolor='white')
            if r['detected']:
                axes[0].text((r['t_start']+r['t_end'])/2, 0, r['label'], ha='center', va='center', fontsize=9, color='white', fontweight='bold')
        axes[0].set_yticks([]); axes[0].set_title('Streaming Detection (green=detected, gray=silence, red=rejected)')
        axes[0].set_xlim(0, full_audio.shape[-1]/16000)

        ts = [(r['t_start']+r['t_end'])/2 for r in results]
        ps = [r['speech_prob'] for r in results]
        axes[1].fill_between(ts, ps, alpha=0.3, color='blue')
        axes[1].plot(ts, ps, 'b-', linewidth=1)
        axes[1].axhline(0.5, color='red', linestyle='--', alpha=0.5, label='VAD threshold')
        axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Speech prob')
        axes[1].set_ylim(0,1); axes[1].set_xlim(0, full_audio.shape[-1]/16000); axes[1].legend(fontsize=8)
        fig.tight_layout(); plt.show()

        detected_n = sum(1 for r in results if r['detected'])
        speech_n   = sum(1 for r in results if r['is_speech'])
        print(f'Segments: {len(results)} | Speech: {speech_n} | Detected: {detected_n}')
    else:
        print('No GSC test files for streaming test')
else:
    print(f'No checkpoint at {best_ckpt_path}')

---
## 12. Copy Best Checkpoint for Demo

In [ ]:
# Find best experiment by AUC on gsc_fixed k=5
best_exp_name, best_auc = None, 0
for exp in EXPERIMENTS:
    key = f'{exp}_gsc_fixed_k5'
    if key in all_results and all_results[key]['auc'] > best_auc:
        best_auc = all_results[key]['auc']
        best_exp_name = exp

if best_exp_name:
    src = DRIVE_PROJECT_P / 'checkpoints' / best_exp_name / 'best.pt'
    dst = Path('checkpoints/best.pt')
    dst.parent.mkdir(exist_ok=True)
    shutil.copy2(str(src), str(dst))
    print(f'Best experiment: {best_exp_name} (AUC={best_auc:.4f})')
    print(f'Copied {src} -> {dst}')
else:
    print('No results found')

# List all checkpoints
for exp in EXPERIMENTS:
    d = DRIVE_PROJECT_P / 'checkpoints' / exp
    if d.exists():
        pts = sorted(d.glob('*.pt'))
        total_mb = sum(f.stat().st_size for f in pts) / 1024 / 1024
        print(f'  {exp}: {len(pts)} files, {total_mb:.1f} MB')

---
## 13. Launch Gradio Demo

In [ ]:
from src.demo.app import load_model, create_app

load_model()
app = create_app()
app.launch(share=True, show_error=True)